# Fine-Tuning Quickstart: QLoRA + PEFT on Google Colab

This notebook fine-tunes **`Qwen/Qwen2.5-1.5B-Instruct`** on the classic **`yahma/alpaca-cleaned`** instruction dataset using **QLoRA** (4-bit quantization) and **LoRA adapters** from the **PEFT** library.

It is designed to run end-to-end on a **free Google Colab T4 GPU**.

**What you will learn:**
1. How to load a base model in 4-bit (QLoRA) to fit on a small GPU.
2. How to attach trainable LoRA adapters with PEFT.
3. How to format an instruction dataset with a chat template.
4. How to train, evaluate (before vs. after), save, and merge adapters.

> **Tip:** In Colab go to **Runtime → Change runtime type → T4 GPU** before running.

## 1. Check the GPU

Make sure a GPU is attached. On the free tier you typically get an NVIDIA **T4 (16 GB)**, which is plenty for QLoRA on a ~1.5B model.

In [ ]:
!nvidia-smi

## 2. Install dependencies

We install the modern PEFT / quantization stack. On Colab `torch` is already present, so we only add the training libraries.

In [ ]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes
print('✅ Dependencies installed. If Colab asks you to restart the runtime, do it and re-run from here.')

## 3. Imports & configuration

All the knobs you might want to tune live in the `CFG` block below.

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel


class CFG:
    # Model & data
    model_name = "Qwen/Qwen2.5-1.5B-Instruct"
    dataset_name = "yahma/alpaca-cleaned"
    num_samples = 2000        # subset for a fast quickstart; raise for better quality
    max_seq_length = 512      # truncate long examples to fit memory

    # LoRA hyper-parameters
    lora_r = 16               # rank of the low-rank update
    lora_alpha = 32           # scaling factor (commonly 2x rank)
    lora_dropout = 0.05
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]

    # Training
    output_dir = "qwen2.5-1.5b-alpaca-lora"
    epochs = 1
    batch_size = 2
    grad_accum = 8            # effective batch size = batch_size * grad_accum = 16
    learning_rate = 2e-4
    seed = 42


# Use bfloat16 if the GPU supports it (Ampere+), otherwise fall back to float16 (T4).
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Compute dtype: {compute_dtype}")
torch.manual_seed(CFG.seed)

## 4. Load the base model in 4-bit (QLoRA)

`BitsAndBytesConfig` quantizes the frozen base weights to **4-bit NF4**. This is what lets a model that would normally need ~6 GB in fp16 fit comfortably alongside the optimizer state on a 16 GB GPU.

- `nf4`: a 4-bit data type designed for normally-distributed weights.
- `double_quant`: quantizes the quantization constants too, saving a bit more memory.
- `compute_dtype`: math is still done in bf16/fp16 for stability.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)
model.config.use_cache = False  # required when using gradient checkpointing
print(model)

## 5. Baseline generation (before fine-tuning)

Let's define a small helper and see how the *base* model answers. We'll re-run this exact prompt after training to compare.

In [ ]:
def generate(prompt, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt}]
    # Newer transformers return a BatchEncoding (dict) here, not a bare tensor,
    # so we keep it as a dict and read input_ids explicitly.
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]

    # Generation needs the KV cache, which is incompatible with gradient
    # checkpointing. Turn checkpointing off + cache on for inference, then
    # restore the training-friendly settings afterwards.
    try:
        was_gc = bool(model.is_gradient_checkpointing)
    except Exception:
        was_gc = False
    if was_gc:
        model.gradient_checkpointing_disable()
    model.config.use_cache = True

    with torch.no_grad():
        out = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,          # break degenerate token loops
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    model.config.use_cache = False
    if was_gc:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)


TEST_PROMPT = "Explain what overfitting is to a 10-year-old, in two sentences."
print("BEFORE fine-tuning:\n")
print(generate(TEST_PROMPT))

## 6. Load & format the dataset

The Alpaca dataset has three fields: `instruction`, `input` (optional context), and `output`. We convert each row into the model's **chat template** so the format matches what the instruct model already expects.

We include the assistant's answer in the text because this is a causal LM objective: the model learns to predict the response tokens that follow the prompt.

In [ ]:
raw = load_dataset(CFG.dataset_name, split="train")
raw = raw.shuffle(seed=CFG.seed).select(range(CFG.num_samples))
print(raw)
print("\nExample row:\n", raw[0])


def to_chat_text(example):
    instruction = example["instruction"].strip()
    context = (example.get("input") or "").strip()
    user = instruction if not context else f"{instruction}\n\n{context}"
    messages = [
        {"role": "user", "content": user},
        {"role": "assistant", "content": example["output"].strip()},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}


formatted = raw.map(to_chat_text, remove_columns=raw.column_names)
print("\nFormatted example:\n")
print(formatted[0]["text"])

## 7. Attach LoRA adapters with PEFT

Instead of updating all ~1.5B parameters, **LoRA** freezes the base model and injects small trainable low-rank matrices (`A` and `B`) into the attention and MLP projection layers. We typically end up training **less than 1%** of the parameters.

`prepare_model_for_kbit_training` does the housekeeping needed for stable 4-bit training (casts layer norms to fp32, enables gradient checkpointing, etc.).

In [ ]:
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    target_modules=CFG.target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Tokenize

We turn the formatted `text` into token IDs. We **don't** pad here — the data collator pads each batch dynamically, which is more efficient. `DataCollatorForLanguageModeling(mlm=False)` also builds the `labels` for us and masks padding tokens with `-100` so they don't contribute to the loss.

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=CFG.max_seq_length,
    )


tokenized = formatted.map(tokenize, remove_columns=formatted.column_names)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(tokenized)

## 9. Training arguments & Trainer

Key choices for a small GPU:
- **`paged_adamw_8bit`**: a memory-efficient 8-bit optimizer.
- **`gradient_checkpointing`**: trades compute for memory.
- **`grad_accum`**: simulates a larger batch size without using more memory.
- A relatively **high learning rate (2e-4)** is normal for LoRA because we train very few parameters.
- **`max_grad_norm=0.3`** (gradient clipping): the standard QLoRA trick to keep fp16 training stable. Without it, the adapter can blow up and the model degenerates into repeating a single token (e.g. `system system system...`).

In [ ]:
args = TrainingArguments(
    output_dir=CFG.output_dir,
    num_train_epochs=CFG.epochs,
    per_device_train_batch_size=CFG.batch_size,
    gradient_accumulation_steps=CFG.grad_accum,
    learning_rate=CFG.learning_rate,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,  # gradient clipping: keeps fp16 QLoRA training stable
    logging_steps=10,
    save_strategy="epoch",
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    seed=CFG.seed,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=collator,
)

## 10. Train

On a free T4 with these settings, ~2,000 samples for 1 epoch usually takes a few minutes. Watch the loss trend downward.

**Memory footprint:** peak VRAM usage during training stays around **~6.2 GB** — well within the T4's 16 GB. You can confirm it live by running `!nvidia-smi` in another cell while training, or check `torch.cuda.max_memory_allocated() / 1e9` afterwards.

In [ ]:
trainer.train()

## 11. Generate again (after fine-tuning)

Same prompt as before. The fine-tuned model should follow the Alpaca-style instruction format more closely / consistently.

In [ ]:
print("AFTER fine-tuning:\n")
print(generate(TEST_PROMPT))

## 12. Save the LoRA adapter

The adapter is tiny (typically a few MB) because it only contains the low-rank matrices, not the full model. You can share just this file and apply it on top of the base model later.

In [ ]:
trainer.model.save_pretrained(CFG.output_dir)
tokenizer.save_pretrained(CFG.output_dir)
print(f"✅ Adapter saved to: {CFG.output_dir}")

# Optional: push to the Hugging Face Hub
# from huggingface_hub import notebook_login
# notebook_login()
# trainer.model.push_to_hub("your-username/qwen2.5-1.5b-alpaca-lora")
# tokenizer.push_to_hub("your-username/qwen2.5-1.5b-alpaca-lora")

## 13. (Optional) Merge the adapter into the base model

For deployment you often want a single standalone model. We reload the base model in **fp16** (not 4-bit), apply the adapter, and call `merge_and_unload()`. This bakes the LoRA update into the weights so inference no longer depends on PEFT.

> This step loads the full-precision model, so it needs more RAM/VRAM. Skip it if you only need the adapter.

In [ ]:
# Free memory from the 4-bit training model first
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    CFG.model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
merged = PeftModel.from_pretrained(base, CFG.output_dir)
merged = merged.merge_and_unload()

merged.save_pretrained("merged-model")
tokenizer.save_pretrained("merged-model")
print("✅ Merged model saved to: merged-model")

## Recap

You just:
1. Loaded a 1.5B model in **4-bit (QLoRA)** to fit on a free GPU.
2. Attached **LoRA adapters** with **PEFT**, training <1% of the parameters.
3. Formatted an instruction dataset with the model's **chat template**.
4. Trained, compared **before vs. after**, saved the adapter, and merged it.

**Where to go next:**
- Increase `num_samples` and `epochs` for better quality.
- Try **completion-only** loss (mask the prompt tokens) for cleaner instruction following.
- Swap in a larger base model (e.g. 7B) — QLoRA still fits on a single 16 GB GPU.
- Use `trl`'s `SFTTrainer` for a higher-level training loop.

Happy fine-tuning!